# HC3 Finance — Dataset 1 & 2 preparation (financial_qa)

Prepares the `financial_qa` bilateral control slice from `v2/data/raw/collected/hc3_finance.jsonl`.

Per `v2/docs/dataset_design_final.md`:
- **§3.2 F / §3.3:** `financial_qa` is a legitimate control slice; no new LLM prompt needed — the HC3
  ChatGPT side serves directly as the LLM side.
- **§5.4:** Use answer text as `text`; preserve `question_id`; maintain human↔chatgpt pairing.
- **§6.1 / §6.11:** `financial_qa` channel = `qa`; no new prompt family; length bins same logic as email.
- **§4.2:** `question_id` is a diagnostic linkage field (not a model feature).

**Schema assignment:**
| field | human | chatgpt |
|-------|-------|---------|
| `label` | 0 | 1 |
| `label_str` | `'human'` | `'llm'` |
| `fraudness` | `'legitimate'` | `'legitimate'` |
| `channel` | `'qa'` | `'qa'` |
| `scenario_family` | `'financial_qa'` | `'financial_qa'` |
| `time_band` | `'legacy'` | `'modern'` |
| `origin_model` | `'human'` | `'openai/chatgpt'` |

**Multi-answer handling:** 570 question_ids have 2 chatgpt answers. Keep the first per `question_id`
(by file order) for reproducibility and 1:1 pairing with the human side.

**Output:** `v2/data/interim/gathered/financial_qa_prepared.jsonl`  
Both human and chatgpt rows in a single file (pairing preserved via `question_id`).

In [1]:
# ── Cell 1: Setup & helpers ───────────────────────────────────────────────
from __future__ import annotations

import re
import sys
import unicodedata
from pathlib import Path

import pandas as pd
from langdetect import detect, LangDetectException, DetectorFactory

DetectorFactory.seed = 0  # reproducibility

BASE     = Path('/Users/askar/projects/antifraud-deepfake-detection/v2')
RAW_PATH = BASE / 'data/raw/collected/hc3_finance.jsonl'
OUT_DIR  = BASE / 'data/interim/gathered'
OUT_PATH = OUT_DIR / 'financial_qa_prepared.jsonl'

sys.path.insert(0, str(BASE))
from config import length_bin as compute_length_bin  # channel-level thresholds from v2/config.py

URL_RE   = re.compile(r'(?i)(?:https?://\S+|www\.\S+|\b[a-z0-9][a-z0-9-]*\.[a-z]{2,}(?:/\S*)?)')
TOKEN_RE = re.compile(r'\w+|[^\w\s]', re.UNICODE)


def normalize_text(text: str) -> str:
    text = '' if text is None else str(text)
    text = unicodedata.normalize('NFKC', text)
    text = text.replace('\uFFFD', ' ')
    text = re.sub(r'[\x00-\x1F\x7F]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text


def mask_url(text: str) -> str:
    # First pass: catch full URLs (http://..., www.domain, bare domain)
    text = URL_RE.sub('[URL]', text)
    # Second pass: catch bare https?:// fragments left when URLs are space-broken
    text = re.sub(r'https?://\S*', '[URL]', text)
    return text


def is_english(text: str) -> bool:
    """Detect English via langdetect (seed=0 for reproducibility)."""
    if not text or len(text.strip()) < 15:
        return False
    try:
        return detect(text[:2000]) == 'en'
    except LangDetectException:
        return False


def to_yes_no(v) -> str:
    return 'yes' if bool(v) else 'no'


def token_length(text: str) -> int:
    return len(TOKEN_RE.findall(text))


print('BASE:    ', BASE)
print('RAW:     ', RAW_PATH)
print('OUT:     ', OUT_PATH)

BASE:     /Users/askar/projects/antifraud-deepfake-detection/v2
RAW:      /Users/askar/projects/antifraud-deepfake-detection/v2/data/raw/collected/hc3_finance.jsonl
OUT:      /Users/askar/projects/antifraud-deepfake-detection/v2/data/interim/gathered/financial_qa_prepared.jsonl


In [2]:
# ── Cell 2: Load & preprocess ─────────────────────────────────────────────
import json

raw_rows = [json.loads(l) for l in RAW_PATH.read_text(encoding='utf-8').splitlines() if l.strip()]
print(f'Total raw rows: {len(raw_rows)}')

# Split by role
human_raw   = [r for r in raw_rows if r.get('role') == 'human']
chatgpt_raw = [r for r in raw_rows if r.get('role') == 'chatgpt']
print(f'  human:   {len(human_raw)}')
print(f'  chatgpt: {len(chatgpt_raw)}')

# For chatgpt: keep first answer per question_id (570 qids have 2 answers)
seen_qids: set[int] = set()
chatgpt_deduped: list[dict] = []
for r in chatgpt_raw:
    qid = r['question_id']
    if qid not in seen_qids:
        seen_qids.add(qid)
        chatgpt_deduped.append(r)
print(f'  chatgpt after per-qid dedup: {len(chatgpt_deduped)}')


def preprocess_rows(rows: list[dict]) -> list[dict]:
    """Normalize, mask URLs, filter English, filter min length."""
    result = []
    for r in rows:
        text_norm   = normalize_text(r['text'])
        if not text_norm:
            continue
        text_masked = normalize_text(mask_url(text_norm))
        if not text_masked:
            continue
        if len(text_masked) < 20:  # min char length gate
            continue
        if not is_english(text_masked):
            continue
        result.append({**r, '_text_masked': text_masked})
    return result


human_proc   = preprocess_rows(human_raw)
chatgpt_proc = preprocess_rows(chatgpt_deduped)
print(f'\nAfter preprocessing:')
print(f'  human:   {len(human_raw)} → {len(human_proc)}')
print(f'  chatgpt: {len(chatgpt_deduped)} → {len(chatgpt_proc)}')

# Dedup by masked text within each side (keep first occurrence)
def dedup_by_text(rows: list[dict]) -> tuple[list[dict], dict[str, int]]:
    seen: dict[str, dict] = {}
    dup_count: dict[str, int] = {}
    for r in rows:
        key = r['_text_masked']
        if key not in seen:
            seen[key] = r
            dup_count[key] = 1
        else:
            dup_count[key] += 1
    return list(seen.values()), dup_count

human_deduped,   human_dup_counts   = dedup_by_text(human_proc)
chatgpt_deduped2, chatgpt_dup_counts = dedup_by_text(chatgpt_proc)
print(f'\nAfter text dedup:')
print(f'  human:   {len(human_proc)} → {len(human_deduped)}')
print(f'  chatgpt: {len(chatgpt_proc)} → {len(chatgpt_deduped2)}')

Total raw rows: 8436
  human:   3933
  chatgpt: 4503
  chatgpt after per-qid dedup: 3933

After preprocessing:
  human:   3933 → 3932
  chatgpt: 3933 → 3932

After text dedup:
  human:   3932 → 3932
  chatgpt: 3932 → 3910


In [3]:
# ── Cell 3: Schema assignment & write ─────────────────────────────────────
out_rows: list[dict] = []


def build_record(
    r: dict,
    dup_counts: dict[str, int],
    label: int,
    label_str: str,
    time_band: str,
    origin_model: str,
) -> dict:
    text = r['_text_masked']
    tok_len  = token_length(text)
    char_len = len(text)
    return {
        'text':            text,
        'label':           label,
        'label_str':       label_str,
        'fraudness':       'legitimate',
        'channel':         'qa',
        'scenario_family': 'financial_qa',
        'source_family':   'hc3_finance',
        'dataset_source':  'hc3_finance.jsonl',
        'question_id':     int(r['question_id']),  # diagnostic linkage field (§4.2)
        'has_url':         to_yes_no(r.get('has_url', False)),
        'char_length':     char_len,
        'token_length':    tok_len,
        'length_bin':      compute_length_bin(tok_len, 'qa'),
        'time_band':       time_band,
        'origin_model':    origin_model,
        'split':           'tbd',
        'lang_filter_method': 'langdetect_v1',
        'n_duplicate_rows': dup_counts.get(text, 1),
    }


for r in human_deduped:
    out_rows.append(build_record(
        r, human_dup_counts,
        label=0, label_str='human',
        time_band='legacy',
        origin_model='human',
    ))

for r in chatgpt_deduped2:
    out_rows.append(build_record(
        r, chatgpt_dup_counts,
        label=1, label_str='llm',
        time_band='modern',
        origin_model='openai/chatgpt',
    ))

import json as _json
OUT_DIR.mkdir(parents=True, exist_ok=True)
with OUT_PATH.open('w', encoding='utf-8') as fh:
    for row in out_rows:
        fh.write(_json.dumps(row, ensure_ascii=False) + '\n')

human_count   = sum(1 for r in out_rows if r['label'] == 0)
chatgpt_count = sum(1 for r in out_rows if r['label'] == 1)
print(f'Written rows:   {len(out_rows)}')
print(f'  human (0):    {human_count}')
print(f'  chatgpt (1):  {chatgpt_count}')
print(f'Output: {OUT_PATH}')

import collections
print('\nlength_bin distribution (human):')
for k, v in sorted(collections.Counter(r['length_bin'] for r in out_rows if r['label'] == 0).items()):
    print(f'  {k}: {v}')
print('length_bin distribution (chatgpt):')
for k, v in sorted(collections.Counter(r['length_bin'] for r in out_rows if r['label'] == 1).items()):
    print(f'  {k}: {v}')

Written rows:   7842
  human (0):    3932
  chatgpt (1):  3910
Output: /Users/askar/projects/antifraud-deepfake-detection/v2/data/interim/gathered/financial_qa_prepared.jsonl

length_bin distribution (human):
  long: 1140
  medium: 1974
  short: 818
length_bin distribution (chatgpt):
  long: 1693
  medium: 2147
  short: 70


In [4]:
# ── Cell 4: Validation ────────────────────────────────────────────────────
check = pd.read_json(OUT_PATH, lines=True)

REQUIRED_FIELDS = {
    'text', 'label', 'label_str', 'fraudness', 'channel',
    'scenario_family', 'source_family', 'dataset_source',
    'time_band', 'length_bin', 'origin_model', 'split',
}
missing = REQUIRED_FIELDS - set(check.columns)
assert not missing, f'Missing required fields: {missing}'

assert check['text'].astype(str).str.strip().ne('').all(), 'empty texts found'
assert set(check['label']).issubset({0, 1}), 'unexpected label values'
assert set(check['label_str']).issubset({'human', 'llm'}), 'unexpected label_str values'
assert (check['channel'] == 'qa').all(), 'unexpected channel'
assert (check['scenario_family'] == 'financial_qa').all(), 'unexpected scenario_family'
assert (check['fraudness'] == 'legitimate').all(), 'unexpected fraudness'
assert set(check['time_band']).issubset({'legacy', 'modern'}), 'unexpected time_band values'
assert set(check['length_bin']).issubset({'short', 'medium', 'long'}), 'unexpected length_bin values'
assert check['text'].str.contains(r'https?://', regex=True).sum() == 0, \
    'some texts contain unmasked URLs'

# Pairing check: each question_id should have exactly 1 human and 1 chatgpt row
hu  = check[check['label'] == 0].set_index('question_id')
cg  = check[check['label'] == 1].set_index('question_id')
paired_qids = set(hu.index) & set(cg.index)
human_only  = set(hu.index) - set(cg.index)
chatgpt_only = set(cg.index) - set(hu.index)
print(f'Paired question_ids (human + chatgpt): {len(paired_qids)}')
print(f'Human-only question_ids:  {len(human_only)}')
print(f'ChatGPT-only question_ids: {len(chatgpt_only)}')

print(f'\nTotal rows: {len(check)}')
print(f'  human (label=0):   {(check["label"]==0).sum()}')
print(f'  chatgpt (label=1): {(check["label"]==1).sum()}')
print(f'\nColumns: {list(check.columns)}')

print('\nlabel distribution:')
print(check['label_str'].value_counts())
print('\ntime_band × label_str:')
print(check.groupby(['time_band', 'label_str']).size().reset_index(name='n').to_string(index=False))
print('\nlength_bin × label_str:')
print(check.groupby(['length_bin', 'label_str']).size().reset_index(name='n').to_string(index=False))

print('\nValidation passed.')

Paired question_ids (human + chatgpt): 3909
Human-only question_ids:  23
ChatGPT-only question_ids: 1

Total rows: 7842
  human (label=0):   3932
  chatgpt (label=1): 3910

Columns: ['text', 'label', 'label_str', 'fraudness', 'channel', 'scenario_family', 'source_family', 'dataset_source', 'question_id', 'has_url', 'char_length', 'token_length', 'length_bin', 'time_band', 'origin_model', 'split', 'lang_filter_method', 'n_duplicate_rows']

label distribution:
label_str
human    3932
llm      3910
Name: count, dtype: int64

time_band × label_str:
time_band label_str    n
   legacy     human 3932
   modern       llm 3910

length_bin × label_str:
length_bin label_str    n
      long     human 1140
      long       llm 1693
    medium     human 1974
    medium       llm 2147
     short     human  818
     short       llm   70

Validation passed.


In [5]:
# ── Cell 5: Summary ───────────────────────────────────────────────────────
check = pd.read_json(OUT_PATH, lines=True)
human_rows   = check[check['label'] == 0]
chatgpt_rows = check[check['label'] == 1]

target_lo, target_hi = 800, 1200

print('── financial_qa_prepared.jsonl summary ──────────────────────────────')
print(f'Total rows:          {len(check)}')
print(f'  human  (label=0):  {len(human_rows)}')
print(f'  chatgpt(label=1):  {len(chatgpt_rows)}')
print(f'Target per side:     {target_lo}–{target_hi}')
print(f'Human in target:     {target_lo <= len(human_rows) <= target_hi}')
print(f'ChatGPT in target:   {target_lo <= len(chatgpt_rows) <= target_hi}')
print()
print('Note: full dataset is above target range. Down-sampling to 800–1200')
print('per side will happen at the Dataset 1 / Dataset 2 assembly stage.')
print()
print('Human token-length percentiles (p25/p50/p75/p90):')
tl_h = human_rows['token_length'].sort_values().values
n = len(tl_h)
print(f'  p25={tl_h[int(n*0.25)]}  p50={tl_h[n//2]}  p75={tl_h[int(n*0.75)]}  p90={tl_h[int(n*0.90)]}')
print()
print('ChatGPT token-length percentiles (p25/p50/p75/p90):')
tl_c = chatgpt_rows['token_length'].sort_values().values
nc = len(tl_c)
print(f'  p25={tl_c[int(nc*0.25)]}  p50={tl_c[nc//2]}  p75={tl_c[int(nc*0.75)]}  p90={tl_c[int(nc*0.90)]}')
print()
print('Files in gathered/:')
for p in sorted(OUT_DIR.glob('*.jsonl')):
    n_lines = sum(1 for _ in p.open())
    print(f'  {p.name:<45s} {n_lines:>5d} rows')

── financial_qa_prepared.jsonl summary ──────────────────────────────
Total rows:          7842
  human  (label=0):  3932
  chatgpt(label=1):  3910
Target per side:     800–1200
Human in target:     False
ChatGPT in target:   False

Note: full dataset is above target range. Down-sampling to 800–1200
per side will happen at the Dataset 1 / Dataset 2 assembly stage.

Human token-length percentiles (p25/p50/p75/p90):
  p25=85  p50=154  p75=273  p90=424

ChatGPT token-length percentiles (p25/p50/p75/p90):
  p25=195  p50=239  p75=284  p90=327

Files in gathered/:
  enron_ham_prepared.jsonl                      15088 rows
  financial_qa_prepared.jsonl                    7842 rows
  mendeley_smishing_prepared.jsonl                369 rows
  nazario_prepared.jsonl                         5201 rows
  nigerian_fraud_prepared.jsonl                  3234 rows
  smishtank_prepared.jsonl                        483 rows
  sms_ham_prepared.jsonl                         4061 rows
  spamassassin_ham_pre